# head

In [1]:
import pickle # Load refs and annotations
import json
import os
import pandas as pd
import numpy as np
import pprint
import json
import cv2
import random

from typing import Any, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.utils.tensorboard import SummaryWriter

import torchvision
import torchvision.transforms as transforms
from torchvision.utils import draw_bounding_boxes
from torchvision import models
import torchmetrics

import pytorch_lightning as pl
from pytorch_lightning.utilities.types import STEP_OUTPUT

from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import CLIPProcessor, CLIPModel

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import clip
from ultralytics import YOLO
from PIL import Image, ImageDraw

from ipywidgets import FloatProgress
import math 
from torch.nn.modules.batchnorm import _BatchNorm
from torchvision.ops import box_convert

In [2]:
device = torch.device('cpu')

In [3]:
clip_model, clip_preprocess = clip.load("RN50", device=device)

In [4]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_area = max(0, x2 - x1 + 1) * max(0, y2 - y1 + 1) # +1 to avoid max(0,0) therefore avoiding 

    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union_area = box1_area + box2_area - intersection_area

    return intersection_area / union_area

In [5]:
class MetricMeter:
    def __init__(self, name="Default", threshold = 0.5, log_dir='./logs/default_run'):
        self.name = name
        self.metrics = []
        self.epochs = []
        self.threshold = threshold
        self.reset()
        self.best_cases_iou = []  # To store the best cases
        self.worst_cases_iou = []
        self.best_cases_sim = []  # To store the best cases
        self.worst_cases_sim = []
        self.writer = SummaryWriter(log_dir=log_dir)

# TODO add SummaryWriter to plot

    def reset(self):
        self.count = 0
        self.iou = 0
        self.epoch = 0
        self.correct_bboxes, self.overall = 0,0
        self.semantic = 0
        self.metrics = []
    
    def add_best_iou(self, item):
        if len(self.best_cases_iou) < 5:
            self.best_cases_iou.append(item)
        else:
            min_best_case = min(self.best_cases_iou, key=lambda x: (x['iou']))
            if item['iou'] > min_best_case['iou']:
                self.best_cases_iou.remove(min_best_case)
                self.best_cases_iou.append(item)
    
    def add_worst_iou(self, item):
        if len(self.worst_cases_iou) < 5:
            self.worst_cases_iou.append(item)
        else:
            max_worst_case = max(self.worst_cases_iou, key=lambda x: (x['iou']))
            if (item['iou'] < max_worst_case['iou']):
                self.worst_cases_iou.remove(max_worst_case)
                self.worst_cases_iou.append(item)
    
    def add_best_confidence(self, item):
        if len(self.best_cases_sim) < 5:
            self.best_cases_sim.append(item)
        else:
            min_best_case = min(self.best_cases_sim, key=lambda x: (x['confidence']))
            if item['confidence'] > min_best_case['confidence']:
                self.best_cases_sim.remove(min_best_case)
                self.best_cases_sim.append(item)
    
    def add_worst_confidence(self, item):
        if len(self.worst_cases_sim) < 5:
            self.worst_cases_sim.append(item)
        else:
            max_worst_case = max(self.worst_cases_sim, key=lambda x: (x['confidence']))
            if (item['confidence'] < max_worst_case['confidence']):
                self.worst_cases_sim.remove(max_worst_case)
                self.worst_cases_sim.append(item)


    def update(self, iou, confidence,filename,bbox_e,bbox_gt):
        self.count += 1
        self.iou += iou
        if(iou >= self.threshold):
            self.correct_bboxes += 1
        self.semantic += confidence

        iteration = {
            'run_loc_acc': self.iou / self.count,
            'run_gro_acc': self.correct_bboxes / self.count,
            'run_sem_acc': self.semantic / self.count,
        }

        item = {
            'iou': iou,
            'confidence':confidence,
            'ground_truth': bbox_gt,
            'candidate': bbox_e,
            'path':filename
        }
        self.metrics.append([iteration])
        self.add_best_confidence(item)
        self.add_worst_confidence(item)
        self.add_best_iou(item)
        self.add_worst_iou(item)
        self.writer.add_scalar('Localization Loss', self.iou / self.count, self.count)
        self.writer.add_scalar('Grounding Accuracy',  self.correct_bboxes / self.count, self.count)
        self.writer.add_scalar('Semantic Similarity', self.semantic / self.count, self.count)

    def new_epoch(self):
        self.epoch += 1
        self.writer.add_scalar('Localization Loss', self.iou / self.count, self.epoch)
        self.writer.add_scalar('Grounding Accuracy',  self.correct_bboxes / self.count, self.epoch)
        self.writer.add_scalar('Semantic Similarity', self.semantic / self.count, self.epoch)
        self.epochs.append(self.metrics,self.best_cases_iou,self.best_cases_sim,self.worst_cases_iou,self.worst_cases_sim)
        self.reset()

    def __repr__(self):
        text = f"{self.name}: {self.avg:.8f}"
        return text
    
    def print_iteration(self):
        print(f"Localization accuracy = {self.iou / self.count}, Grounding Accuracy = {self.correct_bboxes / self.count}, Semantic Similarity = {self.semantic / self.count}")

    def get_best_iou_cases(self):
        return sorted(self.best_cases_iou, key=lambda x: x['iou'], reverse=True)

    def get_worst_iou_cases(self):
        return sorted(self.worst_cases_iou, key=lambda x: x['iou'])

    def get_best_pred_cases(self):
        return sorted(self.best_cases_sim, key=lambda x: x['confidence'], reverse=True)

    def get_worst_pred_cases(self):
        return sorted(self.worst_cases_sim, key=lambda x: x['confidence'])
    
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group["lr"]

In [7]:
with open("./refcocog/annotations/refs(umd).p", "rb") as fp:
  refs = pickle.load(fp)

# 'annotations' will be a dict object mapping the 'annotation_id' to the 'bbox' to make search faster
with open("./refcocog/annotations/instances.json", "rb") as fp:
  data = json.load(fp)
  annotations = dict(sorted({ann["id"]: ann["bbox"] for ann in data["annotations"]}.items()))

In [8]:
def getcaption(elem):
    li = []
    for e in elem["sentences"]:
        li.append(e['raw'])
    return li

In [9]:

class RefCOCOG(Dataset):
    """
    Args:
        The dataset will be the raw data wothput any tipe of preprocessing
        {
            'file_name': 
            'caption':
            'ann_id': needed to extract the relative bbox from the .json file
            'bbox': values are set like following:
                - x 
                - y
                - width 
                - height
        }
    """
    def __init__(self, refs, model, preprocess, annotations, split="train", device = 'cpu', count = 5000):
        
        self.clip_model, self.clip_preprocess = model, preprocess
        self.device = device
        self.images = []
        self.texts = []
        self.filepaths =[]
        self.gt = []
        self.cls = []
        temp = 0
        for elem in [d for d in refs if d["split"]==split]:
            
            # Retrieve Single image
            file_name = os.path.join("./refcocog/images/", f'{"_".join(elem["file_name"].split("_")[:3])}.jpg')
            image = Image.open(file_name)

            # Retrieve possible ground truth measures
            cls = elem['category_id']
            gt = annotations[elem['ann_id']]
            bbox_tnsor = torch.tensor(gt, device=self.device)
            new_bbox = box_convert(bbox_tnsor, 'xywh', 'xyxy')
            # Get all texts related to the picture
            sentences = elem['sentences']
            # for i in sentences:
            self.texts.append(clip.tokenize(sentences[0]['raw']))
            self.images.append(self.clip_preprocess(image))
            self.gt.append(new_bbox)
            self.cls.append(cls)
            self.filepaths.append(file_name)
            # temp += 1
            # if (temp > count):
            #     break

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        images = self.images[idx]
        gt = self.gt[idx]
        cls = self.cls[idx]
        filename = self.filepaths[idx]
        return text, images, gt, cls, filename

    def __call__(self, idx):
        print(json.dumps(self.dataset[idx], indent=4))


In [10]:
class RefCOCOG_noproc(Dataset):
    """
    Args:
        The dataset will be the raw data wothput any tipe of preprocessing
        {
            'file_name': 
            'caption':
            'ann_id': needed to extract the relative bbox from the .json file
            'bbox': values are set like following:
                - x 
                - y
                - width 
                - height
        }
    """
    def __init__(self, refs, model, preprocess, annotations, split="train", device = 'cpu', count = 31):
        
        self.clip_model, self.clip_preprocess = model, preprocess
        self.device = device
        #self.images = []
        self.texts = []
        self.filepaths =[]
        self.gt = []
        self.cls = []
        temp = 0
        for elem in [d for d in refs if d["split"]==split]:
            
            # Retrieve Single image
            file_name = os.path.join("./refcocog/images/", f'{"_".join(elem["file_name"].split("_")[:3])}.jpg')
            #image = Image.open(file_name)

            # Retrieve possible ground truth measures
            cls = elem['category_id']
            gt = annotations[elem['ann_id']]
            bbox_tnsor = torch.tensor(gt, device=self.device)
            new_bbox = box_convert(bbox_tnsor, 'xywh', 'xyxy')
            # Get all texts related to the picture
            sentences = elem['sentences']
            # for i in sentences:
            self.texts.append(clip.tokenize(sentences[0]['raw']))
            #self.images.append(self.clip_preprocess(image))
            self.gt.append(new_bbox)
            self.cls.append(cls)
            self.filepaths.append(file_name)
            temp += 1
            if (temp > count):
                break

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        #images = self.images[idx]
        gt = self.gt[idx]
        cls = self.cls[idx]
        filename = self.filepaths[idx]
        return text, gt, cls, filename#, images

    def __call__(self, idx):
        print(json.dumps(self.dataset[idx], indent=4))


In [11]:
# def pad_image(image):
#     """
#     Performs bottom-right padding of the original image to 640x640 (max size of images in the dataset).
#     Bottom-right padding prevents corruption of bounding boxes.

#     ### Arguments
#     image: a PIL.Image to transform
#     """
#     padded_width, padded_height = 640, 640
#     original_height, original_width = image.shape[:2]
#     bottom_padding = padded_height - original_height
#     right_padding = padded_width - original_width
#     top_padding = 0
#     left_padding = 0
    
#     padded_image = cv2.copyMakeBorder(image, top_padding, bottom_padding, left_padding, right_padding, cv2.BORDER_CONSTANT, value=[0, 0, 0])

#     return padded_image 

# def collate_fn(batch):
#     images = []
#     data = {}

#     #Stores all images in a list
#     for sample in batch:
#         text = sample[0]
#         images = sample[0]
#         gt = sample[0]
#         text = sample[0]
#         image = cv2.imread(sample["file_name"], 3)
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#         image = pad_image(image=image)

#         y_ = image.shape[0]
#         x_ = image.shape[1]


#         images.append(transform(image))

#         data['raw'] = sample['raw']
#         x,y,z,c = sample['bbox'][0:4]
#         y1=y 
#         x1=x 
#         y2=(y + c)
#         x2=(x + z)

#         data['bbox'] = [x,y,x2,y2]
#         data['filename'] = sample["file_name"]
            
#     images = torch.stack(images, dim=0)
#     """
#     for key in batch[0].keys():
#         #if key != "file_name":
#         #    data[key] = [sample[key] for sample in batch]
#         data[key] = [sample[key] for sample in batch]
#         if( key == 'bbox'):
#             x,y,z,c = sample[key][0:4]
#             y1=y 
#             x1=x 
#             y2=(y + z)
#             x2=(x + c)
#     """
#     return images, data

# transform = transforms.Compose([
#     transforms.ToTensor(),
# ])

# dataset

In [12]:
# create dataset and dataloader
print("----------------------Processing train split----------------------------")
dataset_train = RefCOCOG_noproc(refs, clip_model, clip_preprocess, annotations, split="train")
dataloader_train = DataLoader(dataset_train, batch_size=16)
len_train = len(dataset_train)
print(f"Numero esempi in train = {len_train}")

print("----------------------Processing test split-----------------------------")
dataset_test = RefCOCOG_noproc(refs, clip_model, clip_preprocess, annotations, split="test")
dataloader_test = DataLoader(dataset_test, batch_size=16)
len_test = len(dataset_test)
print(f"Numero esempi in train = {len_test}")

print("----------------------Processing eval split-----------------------------")
dataset_eval = RefCOCOG_noproc(refs, clip_model, clip_preprocess, annotations, split="val")
dataloader_eval = DataLoader(dataset_eval, batch_size=16)
len_eval = len(dataset_eval)
print(f"Numero esempi in eval = {len_eval}")

print("------------------------------------------------------------------------")


----------------------Processing train split----------------------------
Numero esempi in train = 32
----------------------Processing test split-----------------------------
Numero esempi in train = 32
----------------------Processing eval split-----------------------------
Numero esempi in eval = 32
------------------------------------------------------------------------


# Val

In [13]:
##########################################################
# YolottoClip - Just for Zero-Shotting the dataset
##########################################################
from  torch.cuda.amp import autocast
MOMENTUM = 0.1 #Default should be 3e-4
class ConvBNReLU(nn.Module):
    '''Module for the Conv-BN-ReLU tuple.'''

    def __init__(self, c_in, c_out, kernel_size, stride, padding, dilation):
        super(ConvBNReLU, self).__init__()
        self.conv = nn.Conv2d(
                c_in, c_out, kernel_size=kernel_size, stride=stride, 
                padding=padding, dilation=dilation, bias=False)
        self.bn = nn.SyncBatchNorm(c_out, momentum=MOMENTUM)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class External_attention(nn.Module):

    '''
    Arguments:
        c (int): The input and output channel number.
    '''
    def __init__(self, c):
        super(External_attention, self).__init__()
        
        self.conv1 = nn.Conv2d(c, c, 1) #Convolution to linear layer
        self.al4 = 0
        self.k = 64
        self.fc0 = ConvBNReLU(2048, 512, 3, 1, 1, 1)
        self.linear_0 = nn.Conv1d(c, self.k, 1, bias=False)
        self.norm_layer = nn.SyncBatchNorm(c, momentum=MOMENTUM)
        self.linear_1 = nn.Conv1d(self.k, c, 1, bias=False)
        self.linear_1.weight.data = self.linear_0.weight.data.permute(1, 0, 2)        
        self.fc1 = nn.Sequential(
            ConvBNReLU(512, 256, 3, 1, 1, 1),
            nn.Dropout2d(p=0.1))
        self.conv2 = nn.Sequential(
            nn.Conv2d(c, c, 1, bias=False),
            self.norm_layer)    
        self.fc2 = nn.Conv2d(256, 80, 1)   
        self.final_linear = nn.Sequential(
            nn.Linear(3920, 1024)
        )
        
        for m in self.modules():
            if isinstance(m, nn.Conv2d): # Kaiming Initialiaztion
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.Conv1d):# He Initialization
                n = m.kernel_size[0] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, _BatchNorm): # BatchNorm Initialization all setted to 1
                m.weight.data.fill_(1)
                if m.bias is not None:
                    m.bias.data.zero_()

    # We need to compute the tensor one example by another in order for the memory units to 
    # grasp possible correlations between the different examples
    def process_features(self, tensor, shift=1):
        cycled_tensor = torch.roll(tensor, shifts=shift, dims=0)
        outputs = []
        
        for i in range(cycled_tensor.shape[0]):
            # Extract each slice along the first dimension
            tensor_slice = cycled_tensor[i]
            
            # Pass the slice through the model's forward method
            output = self.forward(tensor_slice.unsqueeze(0))  # Unsqueeze to add the batch dimension back
            
            # Store the output
            outputs.append(output)
        
        # Combine the outputs back into a single tensor if needed
        combined_output = torch.cat(outputs, dim=0)
        
        return combined_output


    def forward(self, x):
        # print(x.shape)
        x = self.fc0(x)
        idn = x
        x = self.conv1(x)

        b, c, h, w = x.size()
        n = h*w
        x = x.view(b, c, n)   # b * c * n 

        attn = self.linear_0(x) # b, k, n
        attn = F.softmax(attn, dim=-1) # b, k, n

        attn = attn / (1e-9 + attn.sum(dim=1, keepdim=True)) #  # b, k, n
        x = self.linear_1(attn) # b, c, n

        x = x.view(b, c, h, w)
        x = self.conv2(x)
        x = x + idn
        x = F.relu(x)
        x = self.fc1(x)
        x = self.fc2(x)
        x = x.view(1, -1)
        # print(x.shape)
        x = self.final_linear(x)

        return x
    
class ExternalYoloClip(nn.Module):
    def __init__(self, clip_model, clip_preprocess, device = 'cpu'):
        super(ExternalYoloClip, self).__init__()
        #self.yolo = YOLO("yolov8n.pt").eval()
        self.EA = External_attention(512).to(device)
        self.clip_model, self.clip_preprocess = clip_model, clip_preprocess
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        self.processing = []
        self.similarities = []
        self.device = device

    def encode_image(self, image):
        # Encode the image using the CLIP model
        with torch.no_grad():
            image_features = self.clip_model.encode_image(image)

        return image_features
    
    def preprocess_images(self, cropped_images):
        # preprocess with CLIP each cropped PIL image(converts each image in a image
        # of size [3,224,224])
        processing = []
        for image in cropped_images:
            processed_img = self.clip_preprocess(image).to(self.device)
            # proc = self.clip_model.encode_image(processed_img)
            processing.append(processed_img)
        preprocessed = torch.tensor(np.stack(processing))
        # preprocessed = torch.stack(processing).to(self.device) # return a single tensor
        return preprocessed


    def encode_text(self, text):
        with torch.no_grad():
            print("HELL YEAH: ", text.shape)
            text_features = self.clip_model.encode_text(text)
        return text_features
    
    def hook_fn(self, module, input, output):
        self.al = output

    def forward(self, images, text):
        # print(images.shape)
        #bl = 0 # input of layer4 of CLIP's ResNet
        #print(len(images))
        hook_handle = clip_model.visual.layer4.register_forward_hook(self.hook_fn) # handle to retrieve output
        image_features = self.encode_image(images)
        # print(image_features)
        # print(70*'-')
        # # print(self.al.shape)
        with autocast():
            image_features = self.EA.process_features(self.al)
        # print(image_features)
        # print(70*'-')
        # print(image_features.shape)
        # print(70*'-')

        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        with autocast():
            text_features = self.encode_text(text)
        # temp = []
        # temp.append(text_features)
        # temp.append(text_features)
        # temp.append(text_features)
        # text_features2 = torch.cat(temp, dim=0)
        # print(text_features)
        # print(70*'-')
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        # text_features2 = text_features2 / text_features2.norm(dim=-1, keepdim=True)
        logit_scale = self.logit_scale.exp()
        # normalized featuresim
        similarity = (image_features @ text_features.t())
        # print(similarity.cpu())
        # similarity = (image_features @ text_features2.t())
        # print(similarity.cpu())
        scaled_similarity = logit_scale * similarity
        logits_per_image = scaled_similarity
        logits_per_text = scaled_similarity.t()

        # print(text_features.shape)
        # # cosine similarity as logits
        # logits_per_image = logit_scale * (image_features @ text_features.t())
        # print("Logits = " + str(logits_per_image))
        # logits_per_text = logits_per_image.t()

        # shape = [global_batch_size, global_batch_size]
        # hook_handle.remove()

        return logits_per_image, logits_per_text
    
    def evaluator(self, crop_images, text, filepath):
        self.eval()
        best_score = 0
        best_bbox = None
        candidates = []
        images = []

        ex_bbox, cls = self.infer_bboxes(images)

        for bbox in self.infer_bboxes(ex_bbox):
            temp = cv2.imread(filepath)
            image = np.zeros((temp.shape[0], temp.shape[1], temp.shape[2]), dtype=np.uint8)
            image[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])] = temp[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])]
            image = self.clip_preprocess(Image.fromarray(image)).unsqueeze(0).to(device)
            images.append(image)
        
        li,lt = self.model(images,text)
        print(li)


In [14]:
model = ExternalYoloClip(clip_model= clip_model, clip_preprocess=clip_preprocess)

In [15]:
#model.load_state_dict(torch.load())

In [16]:
def infer_bboxes(image_path, yolo):
    yolo = yolo
    #print(image_path)
    results = yolo(image_path, verbose=False)
    # print(results[0])
    bboxes = results[0].boxes.xyxy
    #cls = results[0].boxes.cls
    return bboxes

def get_candidate_crop(text,path,yolo, batch):
    images = []
    texts = []
    best_score = 0
    best_bbox = None
    infered_bboxes = infer_bboxes(path, yolo)
    print("\nLISTA bboxes\n")
    for id, i in enumerate(infered_bboxes):
          print("bbox:", i, " posizione: ", id)
    for bbox in infered_bboxes:
        temp = cv2.imread(path)
        image = np.zeros((temp.shape[0], temp.shape[1], temp.shape[2]), dtype=np.uint8)
        image[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])] = temp[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])]
        image = clip_preprocess(Image.fromarray(image)).to(device)
        #print(image.shape)
        images.append(image)
    imagesa = torch.stack(images)
    texts.append(text)
    texts = torch.stack(texts)

    with torch.no_grad():
            logits_per_image, logits_per_text = model(imagesa, texts)
            matching_score = logits_per_text.cpu().numpy()[0]
            matching_score = np.argmax(matching_score)
            print("id best: ", matching_score)

    if matching_score > best_score:
                best_score = matching_score
                #print("list_bbox:", infered_bboxes[best_score])
                best_bbox = infered_bboxes[best_score]
                best_score +=1
    return best_score, best_bbox


In [17]:
def train_Pipeline(model):
    model.clip_model.train()
    model.EA.train()

def eval_Pipeline(model):
    model.clip_model.eval()
    model.EA.eval()

In [18]:
# cumulative_accuracy = 0.0
# cumulative_loss = 0.0
# overall = 0
# errors = 0
# yolo = YOLO('yolov8n.pt')
# cumulative_iou = 0.0
# cumulative_recall = 0.0
# cumulative_sim = 0.0
# iou_threshold = 0.5
# correct_bboxes = 0


# eval_loop = tqdm(dataloader_eval, position=0, leave=True)
# eval_Pipeline(model)
# with torch.no_grad():
#     for _,data in enumerate(eval_loop):
#         scores = []
#         bboxes = []

#         texts = data[0]
#         texts = texts.squeeze(1).to(device)
#         #images = data[1].to(device)
#         gts = data[1]
#         clss = data[2]
#         filename = data[3]
#         images = []
        
#         for x in filename:
#             temp = Image.open(x)
#             image = clip_preprocess(temp).to(device)
#             #print(image.shape)
#             images.append(image)
            

#         fin = torch.stack(images)
#         batch = len(images)
#         overall += batch

#         for i in range(batch):
#             print()
#             score, ebbox = get_candidate_crop(texts[i], filename[i],yolo, batch)
#             # scores.append(score)
#             print("after get_candidate", ebbox)
#             # bboxes.append(ebbox)
#             iou = compute_iou(ebbox, gts[i])
#             cumulative_iou += iou
#             if(iou > iou_threshold):
#                 #compare_candidate_bbox(filename,bbox_e, bbox_gt)
#                 correct_bboxes += 1
#             similarity = score / 100
#             cumulative_sim += score
#             print("OVERALL: ", overall)
#             loc_acc = cumulative_iou / overall
#             ga = correct_bboxes / overall
#             semsim = cumulative_sim / overall 

        
        
#         eval_loop.set_description(f"Localization accuracy = {loc_acc}, Grounding Accuracy = {ga}, Semantic Similarity = {semsim}")



In [19]:
# model.evaluator(image,bbox,temp)

In [20]:
NUM_EPOCHS = 15
count = 0
iou_threshold = 0.5
loss_meter = MetricMeter(name = 'TrainingEA', threshold=iou_threshold)

running_loc_acc = 0.0
correct_bboxes = 0
overall = 0
semantic = 0
semantic_similarity = 0
running_ga = 0
sem = 0
loc_acc = 0
model = ExternalYoloClip(clip_model,clip_preprocess,device='cpu')

In [21]:
loss_meter.reset()

In [22]:
lr = 0.0001
# wd = 0.002 # best run so far
wd = 0.001
alpha = 1 # to decrease lr over time

In [23]:
cost = nn.CrossEntropyLoss()
optimizer = torch.optim.Adadelta(model.parameters(), lr=lr, weight_decay = wd)
optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=wd)

In [24]:
def freeze_layers(model):
    for param in model.clip_model.parameters():
        param.requires_grad = False

def unfreeze_layers(model):
    for param in model.clip_model.parameters():
        param.requires_grad = True

In [24]:
cumulative_accuracy = 0.0
cumulative_loss = 0.0
overall = 0
errors = 0

test_cumulative_accuracy = 0.0
test_cumulative_loss = 0.0
test_overall = 0
test_errors = 0

for i in range(NUM_EPOCHS):
    loop = tqdm(dataloader_train, position=0, leave=True)
    test_loop = tqdm(dataloader_test, position=0, leave=True)
    train_Pipeline(model)
    for _,data in enumerate(loop):
        try:
        
            texts = data[0]
            texts = texts.squeeze(1).to(device)
            #images = data[1].to(device)
            gts = data[1]
            clss = data[2]
            filename = data[3]
            images = []
            
            for x in filename:
                temp = Image.open(x)
                image = clip_preprocess(temp).to(device)
                images.append(image)

            images = torch.stack(images)
            optimizer.zero_grad()

            # Build Data for training pass

            # since confidence is directly how much bbox and text "resembles" each other 
            li, lt = model.forward(images,texts)

            # Construct the ground truth
            ground_truth = torch.arange(len(images),dtype=torch.long,device=device)
            img_loss = cost(li, ground_truth)
            desc_loss = cost(lt, ground_truth)
            loss = (img_loss + desc_loss)/2
            loss.backward()
            optimizer.step()

            # Keep track of loss and accuracy metrics to see epochs progress
            overall += 16 #batch_size
            cumulative_loss += loss.item()

            _, predicted = li.max(dim=1)
            cumulative_accuracy += predicted.eq(ground_truth).sum().item()
            loss = cumulative_loss / overall 
            acc = cumulative_accuracy / overall
        except:
            errors += 1
            print("diocan un altro " + str(errors))


        loop.set_description(f"Training Epoch {i} values => Loss Iter = {loss}, Accuracy = {acc}")
    
    if( i % 3 == 0):
        eval_Pipeline(model)
        with torch.no_grad():
            for _,data in enumerate(test_loop):
                texts = data[0]
                texts = texts.squeeze(1).to(device)
                #images = data[1].to(device)
                gts = data[1]
                clss = data[2]
                filename = data[3]
                images = []
                
                for x in filename:
                    temp = Image.open(x)
                    image = clip_preprocess(temp).to(device)
                    images.append(image)

                images = torch.stack(images)
                li, lt = model.forward(images,texts)
                ground_truth = torch.arange(len(images),dtype=torch.long,device=device)
                img_loss = cost(li, ground_truth)
                desc_loss = cost(lt, ground_truth)
                loss = (img_loss + desc_loss)/2
                overall += 16 #batch_size
                test_cumulative_loss += loss.item()
                _, predicted = li.max(dim=1)
                test_cumulative_accuracy += predicted.eq(ground_truth).sum().item()
                loss = test_cumulative_loss / overall 
                acc = test_cumulative_accuracy / overall
                test_loop.set_description(f"Testin Epoch {i}. Values => Loss Iter = {loss}, Accuracy = {acc}")


  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

/opt/anaconda3/envs/bagigio/lib/python3.9/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


HELL YEAH:  torch.Size([16, 77])
HELL YEAH:  torch.Size([16, 77])
HELL YEAH:  torch.Size([16, 77])
HELL YEAH:  torch.Size([16, 77])


  0%|          | 0/2 [00:00<?, ?it/s]

...

HELL YEAH:  torch.Size([16, 77])
HELL YEAH:  torch.Size([16, 77])


  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

diocan un altro 1
HELL YEAH:  torch.Size([16, 77])
HELL YEAH:  torch.Size([16, 77])


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), 'saves/EA.pt')

In [25]:
import gc

model.cpu()
del model
gc.collect()
torch.cuda.empty_cache()

# RPN

In [26]:
# class AnchorGenerator(torch.nn.Module):

#   def __init__(self, scales, ratios, num_centers=7, spatial_dim=640, device=None):
#     super().__init__()

#     if device is None:
#       self.device = "cpu" if torch.cuda.is_available () else "cpu"
#     else:
#       self.device = device

#     anchorCenters = self.getAnchorCenters(num_centers, spatial_dim)
#     self.anchors = self.getAnchorBoxes(anchorCenters, scales, ratios, spatial_dim)
#     self.anchors_per_pixel = len(scales) * len(ratios)

#   def forward(self, images, *arg):
#     batch_size = len(images.image_sizes)
#     return [self.anchors] * batch_size

#   def __len__(self):
#     return self.anchors_per_pixel

#   '''
#   Returns the anchor centers evenly spaced along the spatial dimension
#   specified. The parameter 'num_centers' should correspond with the spatial
#   dimension of the feature map.
#   '''
#   def getAnchorCenters(self, num_centers, spatial_dim):
#       interval = math.floor(spatial_dim / num_centers)
#       return torch.arange(0, spatial_dim - interval, interval)

#   '''
#   Starting from the given 'anchorCenters' this function generates all the
#   possible anchor boxes that can be formed combining 'scales' and 'ratios'.

#   Returns tensor [N, K]
#   '''
#   def getAnchorBoxes(self, anchorCenters, scales, ratios, spatial_dim):

#       num_centers = anchorCenters.size(0)
#       num_scales = scales.size(0)
#       num_ratios = ratios.size(0)
#       num_anchors_per_pixel = num_scales * num_ratios

#       # compute the combinations of widths and heights according to the rations
#       # and scales
#       wh_combinations = torch.zeros((num_scales * num_ratios, 2), device=self.device)

#       h_ratios = torch.sqrt(ratios)
#       w_ratios = 1 / h_ratios

#       i = 0
#       for ratio_i in range(ratios.size(0)):
#         for scale in scales:
#           wh_combinations[i,0] = scale * w_ratios[ratio_i] # width
#           wh_combinations[i,1] = scale * h_ratios[ratio_i] # height
#           i += 1

#       wh_combinations = wh_combinations.repeat(num_centers * num_centers,1)

#       # compute the combinations of centers positions
#       centers = torch.zeros((num_centers * num_centers, 2), device=self.device)
#       i = 0
#       for cy in anchorCenters:
#         for cx in anchorCenters:
#           centers[i,0] = cx
#           centers[i,1] = cy
#           i += 1

#       anchors = centers.repeat_interleave(repeats=num_anchors_per_pixel, dim=0)
#       anchors = torch.cat([anchors, wh_combinations], dim=1)
#       anchors_xyxy = box_convert(anchors, 'cxcywh', 'xyxy')

#       return anchors_xyxy.round()

In [27]:
scales = torch.tensor([32, 64, 128, 256, 512])
ratios = torch.tensor([0.5, 1.0, 2.0])

#anchor_generator = AnchorGenerator(scales, ratios, num_centers=7, spatial_dim=640)

In [28]:
# import torchvision.models.detection.rpn as rpn
# class RPNHead(torch.nn.Module):

#     def __init__(self, feature_map_channels, anchors_per_pixel):
#         super().__init__()

#         # The convolutions used to calculate the objectness and the regression offsets
#         self.conv = torchvision.ops.Conv2dNormActivation(feature_map_channels, feature_map_channels, kernel_size=3, activation_layer=torch.nn.ReLU, norm_layer=None)
#         self.cls_logits = torch.nn.Conv2d(feature_map_channels, anchors_per_pixel, kernel_size=1, stride=1)
#         self.reg_offsets = torch.nn.Conv2d(feature_map_channels, anchors_per_pixel * 4, kernel_size=1, stride=1)

#         # Convolutions initialization taken from the original pytorch implementation
#         for layer in self.modules():
#             if isinstance(layer, torch.nn.Conv2d):
#                 torch.nn.init.normal_(layer.weight, std=0.01)
#                 if layer.bias is not None:
#                     #layer.bias.data = layer.bias.data.float()
#                     torch.nn.init.constant_(layer.bias, 0)

        

#     def forward(self, feature_maps):
#         feature_map = feature_maps[0]

#         activated_fm = self.conv(feature_map)
#         cls_logits = self.cls_logits(activated_fm)
#         bbox_reg = self.reg_offsets(activated_fm)

#         return [cls_logits], [bbox_reg]

In [132]:
"""
tools to convert specified type
"""
import torch as t
import numpy as np


def tonumpy(data):
    if isinstance(data, np.ndarray):
        return data
    if isinstance(data, t.Tensor):
        return data.detach().cpu().numpy()


def totensor(data, cuda=True):
    if isinstance(data, np.ndarray):
        tensor = t.from_numpy(data)
    if isinstance(data, t.Tensor):
        tensor = data.detach()
    # if cuda:
    #     #tensor = tensor.cuda()
    #     tensor = tensor
    return tensor


def scalar(data):
    if isinstance(data, np.ndarray):
        return data.reshape(1)[0]
    if isinstance(data, t.Tensor):
        return data.item()

In [133]:
import numpy as np
import numpy as xp

import six
from six import __init__


def loc2bbox(src_bbox, loc):
    """Decode bounding boxes from bounding box offsets and scales.

    Given bounding box offsets and scales computed by
    :meth:`bbox2loc`, this function decodes the representation to
    coordinates in 2D image coordinates.

    Given scales and offsets :math:`t_y, t_x, t_h, t_w` and a bounding
    box whose center is :math:`(y, x) = p_y, p_x` and size :math:`p_h, p_w`,
    the decoded bounding box's center :math:`\\hat{g}_y`, :math:`\\hat{g}_x`
    and size :math:`\\hat{g}_h`, :math:`\\hat{g}_w` are calculated
    by the following formulas.

    * :math:`\\hat{g}_y = p_h t_y + p_y`
    * :math:`\\hat{g}_x = p_w t_x + p_x`
    * :math:`\\hat{g}_h = p_h \\exp(t_h)`
    * :math:`\\hat{g}_w = p_w \\exp(t_w)`

    The decoding formulas are used in works such as R-CNN [#]_.

    The output is same type as the type of the inputs.

    .. [#] Ross Girshick, Jeff Donahue, Trevor Darrell, Jitendra Malik. \
    Rich feature hierarchies for accurate object detection and semantic \
    segmentation. CVPR 2014.

    Args:
        src_bbox (array): A coordinates of bounding boxes.
            Its shape is :math:`(R, 4)`. These coordinates are
            :math:`p_{ymin}, p_{xmin}, p_{ymax}, p_{xmax}`.
        loc (array): An array with offsets and scales.
            The shapes of :obj:`src_bbox` and :obj:`loc` should be same.
            This contains values :math:`t_y, t_x, t_h, t_w`.

    Returns:
        array:
        Decoded bounding box coordinates. Its shape is :math:`(R, 4)`. \
        The second axis contains four values \
        :math:`\\hat{g}_{ymin}, \\hat{g}_{xmin},
        \\hat{g}_{ymax}, \\hat{g}_{xmax}`.

    """

    if src_bbox.shape[0] == 0:
        return xp.zeros((0, 4), dtype=loc.dtype)

    src_bbox = src_bbox.astype(src_bbox.dtype, copy=False)

    src_height = src_bbox[:, 2] - src_bbox[:, 0]
    src_width = src_bbox[:, 3] - src_bbox[:, 1]
    src_ctr_y = src_bbox[:, 0] + 0.5 * src_height
    src_ctr_x = src_bbox[:, 1] + 0.5 * src_width

    dy = loc[:, 0::4]
    dx = loc[:, 1::4]
    dh = loc[:, 2::4]
    dw = loc[:, 3::4]

    ctr_y = dy * src_height[:, xp.newaxis] + src_ctr_y[:, xp.newaxis]
    ctr_x = dx * src_width[:, xp.newaxis] + src_ctr_x[:, xp.newaxis]
    h = xp.exp(dh) * src_height[:, xp.newaxis]
    w = xp.exp(dw) * src_width[:, xp.newaxis]

    dst_bbox = xp.zeros(loc.shape, dtype=loc.dtype)
    dst_bbox[:, 0::4] = ctr_y - 0.5 * h
    dst_bbox[:, 1::4] = ctr_x - 0.5 * w
    dst_bbox[:, 2::4] = ctr_y + 0.5 * h
    dst_bbox[:, 3::4] = ctr_x + 0.5 * w

    return dst_bbox


def bbox2loc(src_bbox, dst_bbox):
    """Encodes the source and the destination bounding boxes to "loc".

    Given bounding boxes, this function computes offsets and scales
    to match the source bounding boxes to the target bounding boxes.
    Mathematcially, given a bounding box whose center is
    :math:`(y, x) = p_y, p_x` and
    size :math:`p_h, p_w` and the target bounding box whose center is
    :math:`g_y, g_x` and size :math:`g_h, g_w`, the offsets and scales
    :math:`t_y, t_x, t_h, t_w` can be computed by the following formulas.

    * :math:`t_y = \\frac{(g_y - p_y)} {p_h}`
    * :math:`t_x = \\frac{(g_x - p_x)} {p_w}`
    * :math:`t_h = \\log(\\frac{g_h} {p_h})`
    * :math:`t_w = \\log(\\frac{g_w} {p_w})`

    The output is same type as the type of the inputs.
    The encoding formulas are used in works such as R-CNN [#]_.

    .. [#] Ross Girshick, Jeff Donahue, Trevor Darrell, Jitendra Malik. \
    Rich feature hierarchies for accurate object detection and semantic \
    segmentation. CVPR 2014.

    Args:
        src_bbox (array): An image coordinate array whose shape is
            :math:`(R, 4)`. :math:`R` is the number of bounding boxes.
            These coordinates are
            :math:`p_{ymin}, p_{xmin}, p_{ymax}, p_{xmax}`.
        dst_bbox (array): An image coordinate array whose shape is
            :math:`(R, 4)`.
            These coordinates are
            :math:`g_{ymin}, g_{xmin}, g_{ymax}, g_{xmax}`.

    Returns:
        array:
        Bounding box offsets and scales from :obj:`src_bbox` \
        to :obj:`dst_bbox`. \
        This has shape :math:`(R, 4)`.
        The second axis contains four values :math:`t_y, t_x, t_h, t_w`.

    """

    height = src_bbox[:, 2] - src_bbox[:, 0]
    width = src_bbox[:, 3] - src_bbox[:, 1]
    ctr_y = src_bbox[:, 0] + 0.5 * height
    ctr_x = src_bbox[:, 1] + 0.5 * width

    base_height = dst_bbox[:, 2] - dst_bbox[:, 0]
    base_width = dst_bbox[:, 3] - dst_bbox[:, 1]
    base_ctr_y = dst_bbox[:, 0] + 0.5 * base_height
    base_ctr_x = dst_bbox[:, 1] + 0.5 * base_width

    eps = xp.finfo(height.dtype).eps
    height = xp.maximum(height, eps)
    width = xp.maximum(width, eps)

    dy = (base_ctr_y - ctr_y) / height
    dx = (base_ctr_x - ctr_x) / width
    dh = xp.log(base_height / height)
    dw = xp.log(base_width / width)

    loc = xp.vstack((dy, dx, dh, dw)).transpose()
    return loc


def bbox_iou(bbox_a, bbox_b):
    """Calculate the Intersection of Unions (IoUs) between bounding boxes.

    IoU is calculated as a ratio of area of the intersection
    and area of the union.

    This function accepts both :obj:`numpy.ndarray` and :obj:`cupy.ndarray` as
    inputs. Please note that both :obj:`bbox_a` and :obj:`bbox_b` need to be
    same type.
    The output is same type as the type of the inputs.

    Args:
        bbox_a (array): An array whose shape is :math:`(N, 4)`.
            :math:`N` is the number of bounding boxes.
            The dtype should be :obj:`numpy.float32`.
        bbox_b (array): An array similar to :obj:`bbox_a`,
            whose shape is :math:`(K, 4)`.
            The dtype should be :obj:`numpy.float32`.

    Returns:
        array:
        An array whose shape is :math:`(N, K)`. \
        An element at index :math:`(n, k)` contains IoUs between \
        :math:`n` th bounding box in :obj:`bbox_a` and :math:`k` th bounding \
        box in :obj:`bbox_b`.

    """
    if bbox_a.shape[1] != 4 or bbox_b.shape[1] != 4:
        raise IndexError

    # top left
    tl = xp.maximum(bbox_a[:, None, :2], bbox_b[:, :2])
    # bottom right
    br = xp.minimum(bbox_a[:, None, 2:], bbox_b[:, 2:])

    area_i = xp.prod(br - tl, axis=2) * (tl < br).all(axis=2)
    area_a = xp.prod(bbox_a[:, 2:] - bbox_a[:, :2], axis=1)
    area_b = xp.prod(bbox_b[:, 2:] - bbox_b[:, :2], axis=1)
    return area_i / (area_a[:, None] + area_b - area_i)


def __test():
    pass


if __name__ == '__main__':
    __test()


def generate_anchor_base(base_size=16, ratios=[0.5, 1, 2],
                         anchor_scales=[8, 16, 32]):
    """Generate anchor base windows by enumerating aspect ratio and scales.

    Generate anchors that are scaled and modified to the given aspect ratios.
    Area of a scaled anchor is preserved when modifying to the given aspect
    ratio.

    :obj:`R = len(ratios) * len(anchor_scales)` anchors are generated by this
    function.
    The :obj:`i * len(anchor_scales) + j` th anchor corresponds to an anchor
    generated by :obj:`ratios[i]` and :obj:`anchor_scales[j]`.

    For example, if the scale is :math:`8` and the ratio is :math:`0.25`,
    the width and the height of the base window will be stretched by :math:`8`.
    For modifying the anchor to the given aspect ratio,
    the height is halved and the width is doubled.

    Args:
        base_size (number): The width and the height of the reference window.
        ratios (list of floats): This is ratios of width to height of
            the anchors.
        anchor_scales (list of numbers): This is areas of anchors.
            Those areas will be the product of the square of an element in
            :obj:`anchor_scales` and the original area of the reference
            window.

    Returns:
        ~numpy.ndarray:
        An array of shape :math:`(R, 4)`.
        Each element is a set of coordinates of a bounding box.
        The second axis corresponds to
        :math:`(y_{min}, x_{min}, y_{max}, x_{max})` of a bounding box.

    """
    py = base_size / 2.
    px = base_size / 2.

    anchor_base = np.zeros((len(ratios) * len(anchor_scales), 4),
                           dtype=np.float32)
    for i in six.moves.range(len(ratios)):
        for j in six.moves.range(len(anchor_scales)):
            h = base_size * anchor_scales[j] * np.sqrt(ratios[i])
            w = base_size * anchor_scales[j] * np.sqrt(1. / ratios[i])

            index = i * len(anchor_scales) + j
            anchor_base[index, 0] = py - h / 2.
            anchor_base[index, 1] = px - w / 2.
            anchor_base[index, 2] = py + h / 2.
            anchor_base[index, 3] = px + w / 2.
    return anchor_base

In [134]:
import numpy as np
import torch
from torchvision.ops import nms
#from model.utils.bbox_tools import bbox2loc, bbox_iou, loc2bbox


class ProposalTargetCreator(object):
    """Assign ground truth bounding boxes to given RoIs.

    The :meth:`__call__` of this class generates training targets
    for each object proposal.
    This is used to train Faster RCNN [#]_.

    .. [#] Shaoqing Ren, Kaiming He, Ross Girshick, Jian Sun. \
    Faster R-CNN: Towards Real-Time Object Detection with \
    Region Proposal Networks. NIPS 2015.

    Args:
        n_sample (int): The number of sampled regions.
        pos_ratio (float): Fraction of regions that is labeled as a
            foreground.
        pos_iou_thresh (float): IoU threshold for a RoI to be considered as a
            foreground.
        neg_iou_thresh_hi (float): RoI is considered to be the background
            if IoU is in
            [:obj:`neg_iou_thresh_hi`, :obj:`neg_iou_thresh_hi`).
        neg_iou_thresh_lo (float): See above.

    """

    def __init__(self,
                 n_sample=128,
                 pos_ratio=0.25, pos_iou_thresh=0.5,
                 neg_iou_thresh_hi=0.5, neg_iou_thresh_lo=0.0
                 ):
        self.n_sample = n_sample
        self.pos_ratio = pos_ratio
        self.pos_iou_thresh = pos_iou_thresh
        self.neg_iou_thresh_hi = neg_iou_thresh_hi
        self.neg_iou_thresh_lo = neg_iou_thresh_lo  # NOTE:default 0.1 in py-faster-rcnn

    def __call__(self, roi, bbox, label,
                 loc_normalize_mean=(0., 0., 0., 0.),
                 loc_normalize_std=(0.1, 0.1, 0.2, 0.2)):
        """Assigns ground truth to sampled proposals.

        This function samples total of :obj:`self.n_sample` RoIs
        from the combination of :obj:`roi` and :obj:`bbox`.
        The RoIs are assigned with the ground truth class labels as well as
        bounding box offsets and scales to match the ground truth bounding
        boxes. As many as :obj:`pos_ratio * self.n_sample` RoIs are
        sampled as foregrounds.

        Offsets and scales of bounding boxes are calculated using
        :func:`model.utils.bbox_tools.bbox2loc`.
        Also, types of input arrays and output arrays are same.

        Here are notations.

        * :math:`S` is the total number of sampled RoIs, which equals \
            :obj:`self.n_sample`.
        * :math:`L` is number of object classes possibly including the \
            background.

        Args:
            roi (array): Region of Interests (RoIs) from which we sample.
                Its shape is :math:`(R, 4)`
            bbox (array): The coordinates of ground truth bounding boxes.
                Its shape is :math:`(R', 4)`.
            label (array): Ground truth bounding box labels. Its shape
                is :math:`(R',)`. Its range is :math:`[0, L - 1]`, where
                :math:`L` is the number of foreground classes.
            loc_normalize_mean (tuple of four floats): Mean values to normalize
                coordinates of bouding boxes.
            loc_normalize_std (tupler of four floats): Standard deviation of
                the coordinates of bounding boxes.

        Returns:
            (array, array, array):

            * **sample_roi**: Regions of interests that are sampled. \
                Its shape is :math:`(S, 4)`.
            * **gt_roi_loc**: Offsets and scales to match \
                the sampled RoIs to the ground truth bounding boxes. \
                Its shape is :math:`(S, 4)`.
            * **gt_roi_label**: Labels assigned to sampled RoIs. Its shape is \
                :math:`(S,)`. Its range is :math:`[0, L]`. The label with \
                value 0 is the background.

        """
        n_bbox, _ = bbox.shape

        roi = np.concatenate((roi, bbox), axis=0)

        pos_roi_per_image = np.round(self.n_sample * self.pos_ratio)
        iou = bbox_iou(roi, bbox)
        gt_assignment = iou.argmax(axis=1)
        max_iou = iou.max(axis=1)
        # Offset range of classes from [0, n_fg_class - 1] to [1, n_fg_class].
        # The label with value 0 is the background.
        gt_roi_label = label[gt_assignment] + 1

        # Select foreground RoIs as those with >= pos_iou_thresh IoU.
        pos_index = np.where(max_iou >= self.pos_iou_thresh)[0]
        pos_roi_per_this_image = int(min(pos_roi_per_image, pos_index.size))
        if pos_index.size > 0:
            pos_index = np.random.choice(
                pos_index, size=pos_roi_per_this_image, replace=False)

        # Select background RoIs as those within
        # [neg_iou_thresh_lo, neg_iou_thresh_hi).
        neg_index = np.where((max_iou < self.neg_iou_thresh_hi) &
                             (max_iou >= self.neg_iou_thresh_lo))[0]
        neg_roi_per_this_image = self.n_sample - pos_roi_per_this_image
        neg_roi_per_this_image = int(min(neg_roi_per_this_image,
                                         neg_index.size))
        if neg_index.size > 0:
            neg_index = np.random.choice(
                neg_index, size=neg_roi_per_this_image, replace=False)

        # The indices that we're selecting (both positive and negative).
        keep_index = np.append(pos_index, neg_index)
        gt_roi_label = gt_roi_label[keep_index]
        gt_roi_label[pos_roi_per_this_image:] = 0  # negative labels --> 0
        sample_roi = roi[keep_index]

        # Compute offsets and scales to match sampled RoIs to the GTs.
        gt_roi_loc = bbox2loc(sample_roi, bbox[gt_assignment[keep_index]])
        gt_roi_loc = ((gt_roi_loc - np.array(loc_normalize_mean, np.float32)
                       ) / np.array(loc_normalize_std, np.float32))

        return sample_roi, gt_roi_loc, gt_roi_label


class AnchorTargetCreator(object):
    """Assign the ground truth bounding boxes to anchors.

    Assigns the ground truth bounding boxes to anchors for training Region
    Proposal Networks introduced in Faster R-CNN [#]_.

    Offsets and scales to match anchors to the ground truth are
    calculated using the encoding scheme of
    :func:`model.utils.bbox_tools.bbox2loc`.

    .. [#] Shaoqing Ren, Kaiming He, Ross Girshick, Jian Sun. \
    Faster R-CNN: Towards Real-Time Object Detection with \
    Region Proposal Networks. NIPS 2015.

    Args:
        n_sample (int): The number of regions to produce.
        pos_iou_thresh (float): Anchors with IoU above this
            threshold will be assigned as positive.
        neg_iou_thresh (float): Anchors with IoU below this
            threshold will be assigned as negative.
        pos_ratio (float): Ratio of positive regions in the
            sampled regions.

    """

    def __init__(self,
                 n_sample=256,
                 pos_iou_thresh=0.7, neg_iou_thresh=0.3,
                 pos_ratio=0.5):
        self.n_sample = n_sample
        self.pos_iou_thresh = pos_iou_thresh
        self.neg_iou_thresh = neg_iou_thresh
        self.pos_ratio = pos_ratio

    def __call__(self, bbox, anchor, img_size):
        """Assign ground truth supervision to sampled subset of anchors.

        Types of input arrays and output arrays are same.

        Here are notations.

        * :math:`S` is the number of anchors.
        * :math:`R` is the number of bounding boxes.

        Args:
            bbox (array): Coordinates of bounding boxes. Its shape is
                :math:`(R, 4)`.
            anchor (array): Coordinates of anchors. Its shape is
                :math:`(S, 4)`.
            img_size (tuple of ints): A tuple :obj:`H, W`, which
                is a tuple of height and width of an image.

        Returns:
            (array, array):

            #NOTE: it's scale not only  offset
            * **loc**: Offsets and scales to match the anchors to \
                the ground truth bounding boxes. Its shape is :math:`(S, 4)`.
            * **label**: Labels of anchors with values \
                :obj:`(1=positive, 0=negative, -1=ignore)`. Its shape \
                is :math:`(S,)`.

        """

        img_H, img_W = img_size

        n_anchor = len(anchor)
        inside_index = _get_inside_index(anchor, img_H, img_W)
        anchor = anchor[inside_index]
        argmax_ious, label = self._create_label(
            inside_index, anchor, bbox)

        # compute bounding box regression targets
        loc = bbox2loc(anchor, bbox[argmax_ious])

        # map up to original set of anchors
        label = _unmap(label, n_anchor, inside_index, fill=-1)
        loc = _unmap(loc, n_anchor, inside_index, fill=0)

        return loc, label

    def _create_label(self, inside_index, anchor, bbox):
        # label: 1 is positive, 0 is negative, -1 is dont care
        label = np.empty((len(inside_index),), dtype=np.int32)
        label.fill(-1)

        argmax_ious, max_ious, gt_argmax_ious = \
            self._calc_ious(anchor, bbox, inside_index)

        # assign negative labels first so that positive labels can clobber them
        label[max_ious < self.neg_iou_thresh] = 0

        # positive label: for each gt, anchor with highest iou
        label[gt_argmax_ious] = 1

        # positive label: above threshold IOU
        label[max_ious >= self.pos_iou_thresh] = 1

        # subsample positive labels if we have too many
        n_pos = int(self.pos_ratio * self.n_sample)
        pos_index = np.where(label == 1)[0]
        if len(pos_index) > n_pos:
            disable_index = np.random.choice(
                pos_index, size=(len(pos_index) - n_pos), replace=False)
            label[disable_index] = -1

        # subsample negative labels if we have too many
        n_neg = self.n_sample - np.sum(label == 1)
        neg_index = np.where(label == 0)[0]
        if len(neg_index) > n_neg:
            disable_index = np.random.choice(
                neg_index, size=(len(neg_index) - n_neg), replace=False)
            label[disable_index] = -1

        return argmax_ious, label

    def _calc_ious(self, anchor, bbox, inside_index):
        # ious between the anchors and the gt boxes
        ious = bbox_iou(anchor, bbox)
        argmax_ious = ious.argmax(axis=1)
        max_ious = ious[np.arange(len(inside_index)), argmax_ious]
        gt_argmax_ious = ious.argmax(axis=0)
        gt_max_ious = ious[gt_argmax_ious, np.arange(ious.shape[1])]
        gt_argmax_ious = np.where(ious == gt_max_ious)[0]

        return argmax_ious, max_ious, gt_argmax_ious


def _unmap(data, count, index, fill=0):
    # Unmap a subset of item (data) back to the original set of items (of
    # size count)

    if len(data.shape) == 1:
        ret = np.empty((count,), dtype=data.dtype)
        ret.fill(fill)
        ret[index] = data
    else:
        ret = np.empty((count,) + data.shape[1:], dtype=data.dtype)
        ret.fill(fill)
        ret[index, :] = data
    return ret


def _get_inside_index(anchor, H, W):
    # Calc indicies of anchors which are located completely inside of the image
    # whose size is speficied.
    index_inside = np.where(
        (anchor[:, 0] >= 0) &
        (anchor[:, 1] >= 0) &
        (anchor[:, 2] <= H) &
        (anchor[:, 3] <= W)
    )[0]
    return index_inside


class ProposalCreator:
    # unNOTE: I'll make it undifferential
    # unTODO: make sure it's ok
    # It's ok
    """Proposal regions are generated by calling this object.

    The :meth:`__call__` of this object outputs object detection proposals by
    applying estimated bounding box offsets
    to a set of anchors.

    This class takes parameters to control number of bounding boxes to
    pass to NMS and keep after NMS.
    If the paramters are negative, it uses all the bounding boxes supplied
    or keep all the bounding boxes returned by NMS.

    This class is used for Region Proposal Networks introduced in
    Faster R-CNN [#]_.

    .. [#] Shaoqing Ren, Kaiming He, Ross Girshick, Jian Sun. \
    Faster R-CNN: Towards Real-Time Object Detection with \
    Region Proposal Networks. NIPS 2015.

    Args:
        nms_thresh (float): Threshold value used when calling NMS.
        n_train_pre_nms (int): Number of top scored bounding boxes
            to keep before passing to NMS in train mode.
        n_train_post_nms (int): Number of top scored bounding boxes
            to keep after passing to NMS in train mode.
        n_test_pre_nms (int): Number of top scored bounding boxes
            to keep before passing to NMS in test mode.
        n_test_post_nms (int): Number of top scored bounding boxes
            to keep after passing to NMS in test mode.
        force_cpu_nms (bool): If this is :obj:`True`,
            always use NMS in CPU mode. If :obj:`False`,
            the NMS mode is selected based on the type of inputs.
        min_size (int): A paramter to determine the threshold on
            discarding bounding boxes based on their sizes.

    """

    def __init__(self,
                 parent_model,
                 nms_thresh=0.7,
                 n_train_pre_nms=12000,
                 n_train_post_nms=2000,
                 n_test_pre_nms=6000,
                 n_test_post_nms=300,
                 min_size=16
                 ):
        self.parent_model = parent_model
        self.nms_thresh = nms_thresh
        self.n_train_pre_nms = n_train_pre_nms
        self.n_train_post_nms = n_train_post_nms
        self.n_test_pre_nms = n_test_pre_nms
        self.n_test_post_nms = n_test_post_nms
        self.min_size = min_size

    def __call__(self, loc, score,
                 anchor, img_size, scale=1.):
        """input should  be ndarray
        Propose RoIs.

        Inputs :obj:`loc, score, anchor` refer to the same anchor when indexed
        by the same index.

        On notations, :math:`R` is the total number of anchors. This is equal
        to product of the height and the width of an image and the number of
        anchor bases per pixel.

        Type of the output is same as the inputs.

        Args:
            loc (array): Predicted offsets and scaling to anchors.
                Its shape is :math:`(R, 4)`.
            score (array): Predicted foreground probability for anchors.
                Its shape is :math:`(R,)`.
            anchor (array): Coordinates of anchors. Its shape is
                :math:`(R, 4)`.
            img_size (tuple of ints): A tuple :obj:`height, width`,
                which contains image size after scaling.
            scale (float): The scaling factor used to scale an image after
                reading it from a file.

        Returns:
            array:
            An array of coordinates of proposal boxes.
            Its shape is :math:`(S, 4)`. :math:`S` is less than
            :obj:`self.n_test_post_nms` in test time and less than
            :obj:`self.n_train_post_nms` in train time. :math:`S` depends on
            the size of the predicted bounding boxes and the number of
            bounding boxes discarded by NMS.

        """
        # NOTE: when test, remember
        # faster_rcnn.eval()
        # to set self.traing = False
        if self.parent_model.training:
            n_pre_nms = self.n_train_pre_nms
            n_post_nms = self.n_train_post_nms
        else:
            n_pre_nms = self.n_test_pre_nms
            n_post_nms = self.n_test_post_nms

        # Convert anchors into proposal via bbox transformations.
        # roi = loc2bbox(anchor, loc)
        roi = loc2bbox(anchor, loc)

        # Clip predicted boxes to image.
        roi[:, slice(0, 4, 2)] = np.clip(
            roi[:, slice(0, 4, 2)], 0, img_size[0])
        roi[:, slice(1, 4, 2)] = np.clip(
            roi[:, slice(1, 4, 2)], 0, img_size[1])

        # Remove predicted boxes with either height or width < threshold.
        min_size = self.min_size * scale
        hs = roi[:, 2] - roi[:, 0]
        ws = roi[:, 3] - roi[:, 1]
        keep = np.where((hs >= min_size) & (ws >= min_size))[0]
        roi = roi[keep, :]
        score = score[keep]

        # Sort all (proposal, score) pairs by score from highest to lowest.
        # Take top pre_nms_topN (e.g. 6000).
        order = score.ravel().argsort()[::-1]
        if n_pre_nms > 0:
            order = order[:n_pre_nms]
        roi = roi[order, :]
        score = score[order]

        # Apply nms (e.g. threshold = 0.7).
        # Take after_nms_topN (e.g. 300).

        # unNOTE: somthing is wrong here!
        # TODO: remove cuda.to_gpu
        keep = nms(
            torch.from_numpy(roi),
            torch.from_numpy(score),
            self.nms_thresh)
        # keep = nms(
        #     torch.from_numpy(roi).cuda(),
        #     torch.from_numpy(score).cuda(),
        #     self.nms_thresh)
        if n_post_nms > 0:
            keep = keep[:n_post_nms]
        roi = roi[keep.cpu().numpy()]
        return roi

In [135]:
import numpy as np
from torch.nn import functional as F
import torch as t
from torch import nn

# from model.utils.bbox_tools import generate_anchor_base
# from model.utils.creator_tool import ProposalCreator


class RegionProposalNetwork(nn.Module):
    """Region Proposal Network introduced in Faster R-CNN.

    This is Region Proposal Network introduced in Faster R-CNN [#]_.
    This takes features extracted from images and propose
    class agnostic bounding boxes around "objects".

    .. [#] Shaoqing Ren, Kaiming He, Ross Girshick, Jian Sun. \
    Faster R-CNN: Towards Real-Time Object Detection with \
    Region Proposal Networks. NIPS 2015.

    Args:
        in_channels (int): The channel size of input.
        mid_channels (int): The channel size of the intermediate tensor.
        ratios (list of floats): This is ratios of width to height of
            the anchors.
        anchor_scales (list of numbers): This is areas of anchors.
            Those areas will be the product of the square of an element in
            :obj:`anchor_scales` and the original area of the reference
            window.
        feat_stride (int): Stride size after extracting features from an
            image.
        initialW (callable): Initial weight value. If :obj:`None` then this
            function uses Gaussian distribution scaled by 0.1 to
            initialize weight.
            May also be a callable that takes an array and edits its values.
        proposal_creator_params (dict): Key valued paramters for
            :class:`model.utils.creator_tools.ProposalCreator`.

    .. seealso::
        :class:`~model.utils.creator_tools.ProposalCreator`

    """

    def __init__(
            self, in_channels=2048, mid_channels=1024, ratios=[0.5, 1, 2],
            anchor_scales=[8, 16, 32], feat_stride=16,
            proposal_creator_params=dict(),
    ):
        super(RegionProposalNetwork, self).__init__()
        self.anchor_base = generate_anchor_base(
            anchor_scales=anchor_scales, ratios=ratios)
        self.feat_stride = feat_stride
        self.proposal_layer = ProposalCreator(self, **proposal_creator_params)
        n_anchor = self.anchor_base.shape[0]
        self.conv1 = nn.Conv2d(in_channels, mid_channels, 3, 1, 1)
        self.score = nn.Conv2d(mid_channels, n_anchor * 2, 1, 1, 0)
        self.loc = nn.Conv2d(mid_channels, n_anchor * 4, 1, 1, 0)
        normal_init(self.conv1, 0, 0.01)
        normal_init(self.score, 0, 0.01)
        normal_init(self.loc, 0, 0.01)

    def forward(self, x, img_size, scale=1.):
        """Forward Region Proposal Network.

        Here are notations.

        * :math:`N` is batch size.
        * :math:`C` channel size of the input.
        * :math:`H` and :math:`W` are height and witdh of the input feature.
        * :math:`A` is number of anchors assigned to each pixel.

        Args:
            x (~torch.autograd.Variable): The Features extracted from images.
                Its shape is :math:`(N, C, H, W)`.
            img_size (tuple of ints): A tuple :obj:`height, width`,
                which contains image size after scaling.
            scale (float): The amount of scaling done to the input images after
                reading them from files.

        Returns:
            (~torch.autograd.Variable, ~torch.autograd.Variable, array, array, array):

            This is a tuple of five following values.

            * **rpn_locs**: Predicted bounding box offsets and scales for \
                anchors. Its shape is :math:`(N, H W A, 4)`.
            * **rpn_scores**:  Predicted foreground scores for \
                anchors. Its shape is :math:`(N, H W A, 2)`.
            * **rois**: A bounding box array containing coordinates of \
                proposal boxes.  This is a concatenation of bounding box \
                arrays from multiple images in the batch. \
                Its shape is :math:`(R', 4)`. Given :math:`R_i` predicted \
                bounding boxes from the :math:`i` th image, \
                :math:`R' = \\sum _{i=1} ^ N R_i`.
            * **roi_indices**: An array containing indices of images to \
                which RoIs correspond to. Its shape is :math:`(R',)`.
            * **anchor**: Coordinates of enumerated shifted anchors. \
                Its shape is :math:`(H W A, 4)`.

        """
        n, _, hh, ww = x.shape
        anchor = _enumerate_shifted_anchor(
            np.array(self.anchor_base),
            self.feat_stride, hh, ww)

        n_anchor = anchor.shape[0] // (hh * ww)
        h = F.relu(self.conv1(x))

        rpn_locs = self.loc(h)
        # UNNOTE: check whether need contiguous
        # A: Yes
        rpn_locs = rpn_locs.permute(0, 2, 3, 1).contiguous().view(n, -1, 4)
        rpn_scores = self.score(h)
        rpn_scores = rpn_scores.permute(0, 2, 3, 1).contiguous()
        rpn_softmax_scores = F.softmax(rpn_scores.view(n, hh, ww, n_anchor, 2), dim=4)
        rpn_fg_scores = rpn_softmax_scores[:, :, :, :, 1].contiguous()
        rpn_fg_scores = rpn_fg_scores.view(n, -1)
        rpn_scores = rpn_scores.view(n, -1, 2)

        rois = list()
        roi_indices = list()
        for i in range(n):
            roi = self.proposal_layer(
                rpn_locs[i].cpu().data.numpy(),
                rpn_fg_scores[i].cpu().data.numpy(),
                anchor, img_size,
                scale=scale)
            batch_index = i * np.ones((len(roi),), dtype=np.int32)
            rois.append(roi)
            roi_indices.append(batch_index)

        rois = np.concatenate(rois, axis=0)
        roi_indices = np.concatenate(roi_indices, axis=0)
        return rpn_locs, rpn_scores, rois, roi_indices, anchor


def _enumerate_shifted_anchor(anchor_base, feat_stride, height, width):
    # Enumerate all shifted anchors:
    #
    # add A anchors (1, A, 4) to
    # cell K shifts (K, 1, 4) to get
    # shift anchors (K, A, 4)
    # reshape to (K*A, 4) shifted anchors
    # return (K*A, 4)

    # !TODO: add support for torch.CudaTensor
    # xp = cuda.get_array_module(anchor_base)
    # it seems that it can't be boosed using GPU
    import numpy as xp
    shift_y = xp.arange(0, height * feat_stride, feat_stride)
    shift_x = xp.arange(0, width * feat_stride, feat_stride)
    shift_x, shift_y = xp.meshgrid(shift_x, shift_y)
    shift = xp.stack((shift_y.ravel(), shift_x.ravel(),
                      shift_y.ravel(), shift_x.ravel()), axis=1)

    A = anchor_base.shape[0]
    K = shift.shape[0]
    anchor = anchor_base.reshape((1, A, 4)) + \
             shift.reshape((1, K, 4)).transpose((1, 0, 2))
    anchor = anchor.reshape((K * A, 4)).astype(np.float32)
    return anchor


def _enumerate_shifted_anchor_torch(anchor_base, feat_stride, height, width):
    # Enumerate all shifted anchors:
    #
    # add A anchors (1, A, 4) to
    # cell K shifts (K, 1, 4) to get
    # shift anchors (K, A, 4)
    # reshape to (K*A, 4) shifted anchors
    # return (K*A, 4)

    # !TODO: add support for torch.CudaTensor
    # xp = cuda.get_array_module(anchor_base)
    import torch as t
    shift_y = t.arange(0, height * feat_stride, feat_stride)
    shift_x = t.arange(0, width * feat_stride, feat_stride)
    shift_x, shift_y = xp.meshgrid(shift_x, shift_y)
    shift = xp.stack((shift_y.ravel(), shift_x.ravel(),
                      shift_y.ravel(), shift_x.ravel()), axis=1)

    A = anchor_base.shape[0]
    K = shift.shape[0]
    anchor = anchor_base.reshape((1, A, 4)) + \
             shift.reshape((1, K, 4)).transpose((1, 0, 2))
    anchor = anchor.reshape((K * A, 4)).astype(np.float32)
    return anchor


def normal_init(m, mean, stddev, truncated=False):
    """
    weight initalizer: truncated normal and random normal.
    """
    # x is a parameter
    if truncated:
        m.weight.data.normal_().fmod_(2).mul_(stddev).add_(mean)  # not a perfect approximation
    else:
        m.weight.data.normal_(mean, stddev)
        m.bias.data.zero_()

In [136]:
from __future__ import  absolute_import
from __future__ import division
import torch as t
import numpy as np
#from utils import array_tool as at
#from model.utils.bbox_tools import loc2bbox
from torchvision.ops import nms
# from model.utils.nms import non_maximum_suppression

from torch import nn
#from data.dataset import preprocess
from torch.nn import functional as F
#from utils.config import opt


def nograd(f):
    def new_f(*args,**kwargs):
        with t.no_grad():
           return f(*args,**kwargs)
    return new_f

class FasterRCNN(nn.Module):
    """Base class for Faster R-CNN.

    This is a base class for Faster R-CNN links supporting object detection
    API [#]_. The following three stages constitute Faster R-CNN.

    1. **Feature extraction**: Images are taken and their \
        feature maps are calculated.
    2. **Region Proposal Networks**: Given the feature maps calculated in \
        the previous stage, produce set of RoIs around objects.
    3. **Localization and Classification Heads**: Using feature maps that \
        belong to the proposed RoIs, classify the categories of the objects \
        in the RoIs and improve localizations.

    Each stage is carried out by one of the callable
    :class:`torch.nn.Module` objects :obj:`feature`, :obj:`rpn` and :obj:`head`.

    There are two functions :meth:`predict` and :meth:`__call__` to conduct
    object detection.
    :meth:`predict` takes images and returns bounding boxes that are converted
    to image coordinates. This will be useful for a scenario when
    Faster R-CNN is treated as a black box function, for instance.
    :meth:`__call__` is provided for a scnerario when intermediate outputs
    are needed, for instance, for training and debugging.

    Links that support obejct detection API have method :meth:`predict` with
    the same interface. Please refer to :meth:`predict` for
    further details.

    .. [#] Shaoqing Ren, Kaiming He, Ross Girshick, Jian Sun. \
    Faster R-CNN: Towards Real-Time Object Detection with \
    Region Proposal Networks. NIPS 2015.

    Args:
        extractor (nn.Module): A module that takes a BCHW image
            array and returns feature maps.
        rpn (nn.Module): A module that has the same interface as
            :class:`model.region_proposal_network.RegionProposalNetwork`.
            Please refer to the documentation found there.
        head (nn.Module): A module that takes
            a BCHW variable, RoIs and batch indices for RoIs. This returns class
            dependent localization paramters and class scores.
        loc_normalize_mean (tuple of four floats): Mean values of
            localization estimates.
        loc_normalize_std (tupler of four floats): Standard deviation
            of localization estimates.

    """

    def __init__(self, extractor, rpn, head,
                loc_normalize_mean = (0., 0., 0., 0.),
                loc_normalize_std = (0.1, 0.1, 0.2, 0.2)
    ):
        super(FasterRCNN, self).__init__()
        self.extractor = extractor
        self.rpn = rpn
        self.head = head

        # mean and std
        self.loc_normalize_mean = loc_normalize_mean
        self.loc_normalize_std = loc_normalize_std
        self.use_preset('evaluate')

    @property
    def n_class(self):
        # Total number of classes including the background.
        return self.head.n_class

    def forward(self, x, scale=1.):
        """Forward Faster R-CNN.

        Scaling paramter :obj:`scale` is used by RPN to determine the
        threshold to select small objects, which are going to be
        rejected irrespective of their confidence scores.

        Here are notations used.

        * :math:`N` is the number of batch size
        * :math:`R'` is the total number of RoIs produced across batches. \
            Given :math:`R_i` proposed RoIs from the :math:`i` th image, \
            :math:`R' = \\sum _{i=1} ^ N R_i`.
        * :math:`L` is the number of classes excluding the background.

        Classes are ordered by the background, the first class, ..., and
        the :math:`L` th class.

        Args:
            x (autograd.Variable): 4D image variable.
            scale (float): Amount of scaling applied to the raw image
                during preprocessing.

        Returns:
            Variable, Variable, array, array:
            Returns tuple of four values listed below.

            * **roi_cls_locs**: Offsets and scalings for the proposed RoIs. \
                Its shape is :math:`(R', (L + 1) \\times 4)`.
            * **roi_scores**: Class predictions for the proposed RoIs. \
                Its shape is :math:`(R', L + 1)`.
            * **rois**: RoIs proposed by RPN. Its shape is \
                :math:`(R', 4)`.
            * **roi_indices**: Batch indices of RoIs. Its shape is \
                :math:`(R',)`.

        """
        img_size = x.shape[2:]

        h = self.extractor(x)
        print("HELLO: ", h)
        rpn_locs, rpn_scores, rois, roi_indices, anchor = \
            self.rpn(h, img_size, scale)
        
        roi_cls_locs, roi_scores = self.head(
            h, rois, roi_indices)
        return roi_cls_locs, roi_scores, rois, roi_indices

    def use_preset(self, preset):
        """Use the given preset during prediction.

        This method changes values of :obj:`self.nms_thresh` and
        :obj:`self.score_thresh`. These values are a threshold value
        used for non maximum suppression and a threshold value
        to discard low confidence proposals in :meth:`predict`,
        respectively.

        If the attributes need to be changed to something
        other than the values provided in the presets, please modify
        them by directly accessing the public attributes.

        Args:
            preset ({'visualize', 'evaluate'): A string to determine the
                preset to use.

        """
        if preset == 'visualize':
            self.nms_thresh = 0.3
            self.score_thresh = 0.7
        elif preset == 'evaluate':
            self.nms_thresh = 0.3
            self.score_thresh = 0.05
        else:
            raise ValueError('preset must be visualize or evaluate')

    def _suppress(self, raw_cls_bbox, raw_prob):
        bbox = list()
        label = list()
        score = list()
        # skip cls_id = 0 because it is the background class
        for l in range(1, self.n_class):
            cls_bbox_l = raw_cls_bbox.reshape((-1, self.n_class, 4))[:, l, :]
            prob_l = raw_prob[:, l]
            mask = prob_l > self.score_thresh
            cls_bbox_l = cls_bbox_l[mask]
            prob_l = prob_l[mask]
            keep = nms(cls_bbox_l, prob_l,self.nms_thresh)
            # import ipdb;ipdb.set_trace()
            # keep = cp.asnumpy(keep)
            bbox.append(cls_bbox_l[keep].cpu().numpy())
            # The labels are in [0, self.n_class - 2].
            label.append((l - 1) * np.ones((len(keep),)))
            score.append(prob_l[keep].cpu().numpy())
        bbox = np.concatenate(bbox, axis=0).astype(np.float32)
        label = np.concatenate(label, axis=0).astype(np.int32)
        score = np.concatenate(score, axis=0).astype(np.float32)
        return bbox, label, score

    @nograd
    def predict(self, imgs,sizes=None,visualize=False):
        """Detect objects from images.

        This method predicts objects for each image.

        Args:
            imgs (iterable of numpy.ndarray): Arrays holding images.
                All images are in CHW and RGB format
                and the range of their value is :math:`[0, 255]`.

        Returns:
           tuple of lists:
           This method returns a tuple of three lists,
           :obj:`(bboxes, labels, scores)`.

           * **bboxes**: A list of float arrays of shape :math:`(R, 4)`, \
               where :math:`R` is the number of bounding boxes in a image. \
               Each bouding box is organized by \
               :math:`(y_{min}, x_{min}, y_{max}, x_{max})` \
               in the second axis.
           * **labels** : A list of integer arrays of shape :math:`(R,)`. \
               Each value indicates the class of the bounding box. \
               Values are in range :math:`[0, L - 1]`, where :math:`L` is the \
               number of the foreground classes.
           * **scores** : A list of float arrays of shape :math:`(R,)`. \
               Each value indicates how confident the prediction is.

        """
        self.eval()
        if visualize:
            self.use_preset('visualize')
            prepared_imgs = list()
            sizes = list()
            for img in imgs:
                size = img.shape[1:]
                img = preprocess(tonumpy(img))
                prepared_imgs.append(img)
                sizes.append(size)
        else:
             prepared_imgs = imgs 
        bboxes = list()
        labels = list()
        scores = list()
        for img, size in zip(prepared_imgs, sizes):
            img = totensor(img[None]).float()
            scale = img.shape[3] / size[1]
            roi_cls_loc, roi_scores, rois, _ = self(img, scale=scale)
            # We are assuming that batch size is 1.
            roi_score = roi_scores.data
            roi_cls_loc = roi_cls_loc.data
            roi = totensor(rois) / scale

            # Convert predictions to bounding boxes in image coordinates.
            # Bounding boxes are scaled to the scale of the input images.
            # mean = t.Tensor(self.loc_normalize_mean).cuda(). \
            #     repeat(self.n_class)[None]
            # std = t.Tensor(self.loc_normalize_std).cuda(). \
            #     repeat(self.n_class)[None]
            
            mean = t.Tensor(self.loc_normalize_mean). \
                repeat(self.n_class)[None]
            std = t.Tensor(self.loc_normalize_std). \
                repeat(self.n_class)[None]

            roi_cls_loc = (roi_cls_loc * std + mean)
            roi_cls_loc = roi_cls_loc.view(-1, self.n_class, 4)
            roi = roi.view(-1, 1, 4).expand_as(roi_cls_loc)
            cls_bbox = loc2bbox(tonumpy(roi).reshape((-1, 4)),
                                tonumpy(roi_cls_loc).reshape((-1, 4)))
            cls_bbox = totensor(cls_bbox)
            cls_bbox = cls_bbox.view(-1, self.n_class * 4)
            # clip bounding box
            cls_bbox[:, 0::2] = (cls_bbox[:, 0::2]).clamp(min=0, max=size[0])
            cls_bbox[:, 1::2] = (cls_bbox[:, 1::2]).clamp(min=0, max=size[1])

            prob = (F.softmax(totensor(roi_score), dim=1))

            bbox, label, score = self._suppress(cls_bbox, prob)
            bboxes.append(bbox)
            labels.append(label)
            scores.append(score)

        self.use_preset('evaluate')
        self.train()
        return bboxes, labels, scores

    def get_optimizer(self):
        """
        return optimizer, It could be overwriten if you want to specify 
        special optimizer
        """
        lr = opt.lr
        params = []
        for key, value in dict(self.named_parameters()).items():
            if value.requires_grad:
                if 'bias' in key:
                    params += [{'params': [value], 'lr': lr * 2, 'weight_decay': 0}]
                else:
                    params += [{'params': [value], 'lr': lr, 'weight_decay': opt.weight_decay}]
        if opt.use_adam:
            self.optimizer = t.optim.Adam(params)
        else:
            self.optimizer = t.optim.SGD(params, momentum=0.9)
        return self.optimizer

    def scale_lr(self, decay=0.1):
        for param_group in self.optimizer.param_groups:
            param_group['lr'] *= decay
        return self.optimizer

In [137]:
from torchvision.ops import RoIPool

In [147]:
class VGG16RoIHead(nn.Module):
    """Faster R-CNN Head for VGG-16 based implementation.
    This class is used as a head for Faster R-CNN.
    This outputs class-wise localizations and classification based on feature
    maps in the given RoIs.
    
    Args:
        n_class (int): The number of classes possibly including the background.
        roi_size (int): Height and width of the feature maps after RoI-pooling.
        spatial_scale (float): Scale of the roi is resized.
        classifier (nn.Module): Two layer Linear ported from vgg16

    """
    def __init__(self, n_class, roi_size, spatial_scale,
                 classifier):
    # def __init__(self, n_class, roi_size, spatial_scale):
        # n_class includes the background
        super(VGG16RoIHead, self).__init__()

        self.classifier = classifier
        self.cls_loc = nn.Linear(4096, n_class * 4)
        self.score = nn.Linear(4096, n_class)

        normal_init(self.cls_loc, 0, 0.001)
        normal_init(self.score, 0, 0.01)

        self.n_class = n_class
        self.roi_size = roi_size
        self.spatial_scale = spatial_scale
        self.roi = RoIPool( (self.roi_size, self.roi_size),self.spatial_scale)

    def forward(self, x, rois, roi_indices):
        """Forward the chain.

        We assume that there are :math:`N` batches.

        Args:
            x (Variable): 4D image variable.
            rois (Tensor): A bounding box array containing coordinates of
                proposal boxes.  This is a concatenation of bounding box
                arrays from multiple images in the batch.
                Its shape is :math:`(R', 4)`. Given :math:`R_i` proposed
                RoIs from the :math:`i` th image,
                :math:`R' = \\sum _{i=1} ^ N R_i`.
            roi_indices (Tensor): An array containing indices of images to
                which bounding boxes correspond to. Its shape is :math:`(R',)`.

        """
        # in case roi_indices is  ndarray
        roi_indices = totensor(roi_indices).float()
        rois = totensor(rois).float()
        print("SHAPE", roi_indices.shape)
        print(rois.shape)
        indices_and_rois = t.cat([roi_indices[:, None], rois], dim=1)
        # NOTE: important: yx->xy
        xy_indices_and_rois = indices_and_rois[:, [0, 2, 1, 4, 3]]
        indices_and_rois =  xy_indices_and_rois.contiguous()

        pool = self.roi(x, indices_and_rois)
        pool = pool.view(pool.size(0), -1)
        #return pool
        fc7 = self.classifier(pool)
        roi_cls_locs = self.cls_loc(fc7)
        roi_scores = self.score(fc7)
        return roi_cls_locs, roi_scores


def normal_init(m, mean, stddev, truncated=False):
    """
    weight initalizer: truncated normal and random normal.
    """
    # x is a parameter
    if truncated:
        m.weight.data.normal_().fmod_(2).mul_(stddev).add_(mean)  # not a perfect approximation
    else:
        m.weight.data.normal_(mean, stddev)
        m.bias.data.zero_()

In [148]:
# self.faster_rcnn = faster_rcnn
# self.rpn_sigma = opt.rpn_sigma
# self.roi_sigma = opt.roi_sigma

# # target creator create gt_bbox gt_label etc as training targets. 
# self.anchor_target_creator = AnchorTargetCreator()
# self.proposal_target_creator = ProposalTargetCreator()

# self.loc_normalize_mean = faster_rcnn.loc_normalize_mean
# self.loc_normalize_std = faster_rcnn.loc_normalize_std

In [149]:
class ImageList:
    def __init__(self, image_sizes):
        self.image_sizes = image_sizes

def getLoss(cls_loss, reg_loss, lambda_rpn=1):
    return cls_loss + lambda_rpn * reg_loss

def computeSumIou(gtBoxes, predictedBoxes):
  iouMatrix = torchvision.ops.box_iou(gtBoxes, predictedBoxes).diag()
  return iouMatrix.sum()

In [150]:
##########################################################
# YolottoClip - Just for Zero-Shotting the dataset
##########################################################
# from torch.amp import autocast
MOMENTUM = 0.1 #Default should be 3e-4
class ConvBNReLU(nn.Module):
    '''Module for the Conv-BN-ReLU tuple.'''

    def __init__(self, c_in, c_out, kernel_size, stride, padding, dilation):
        super(ConvBNReLU, self).__init__()
        self.conv = nn.Conv2d(
                c_in, c_out, kernel_size=kernel_size, stride=stride, 
                padding=padding, dilation=dilation, bias=False)
        self.bn = nn.SyncBatchNorm(c_out, momentum=MOMENTUM)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class External_attention(nn.Module):

    '''
    Arguments:
        c (int): The input and output channel number.
    '''
    def __init__(self, c):
        super(External_attention, self).__init__()
        
        self.conv1 = nn.Conv2d(c, c, 1) #Convolution to linear layer
        self.al4 = 0
        self.k = 64
        self.fc0 = ConvBNReLU(2048, 512, 3, 1, 1, 1)
        self.linear_0 = nn.Conv1d(c, self.k, 1, bias=False)
        self.norm_layer = nn.SyncBatchNorm(c, momentum=MOMENTUM)
        self.linear_1 = nn.Conv1d(self.k, c, 1, bias=False)
        self.linear_1.weight.data = self.linear_0.weight.data.permute(1, 0, 2)        
        self.fc1 = nn.Sequential(
            ConvBNReLU(512, 256, 3, 1, 1, 1),
            nn.Dropout2d(p=0.1))
        self.conv2 = nn.Sequential(
            nn.Conv2d(c, c, 1, bias=False),
            self.norm_layer)    
        self.fc2 = nn.Conv2d(256, 80, 1)   
        self.final_linear = nn.Sequential(
            nn.Linear(3920, 1024)
        )
        
        for m in self.modules():
            if isinstance(m, nn.Conv2d): # Kaiming Initialiaztion
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.Conv1d):# He Initialization
                n = m.kernel_size[0] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, _BatchNorm): # BatchNorm Initialization all setted to 1
                m.weight.data.fill_(1)
                if m.bias is not None:
                    m.bias.data.zero_()

    def process_features(self, tensor, shift=1):
        cycled_tensor = torch.roll(tensor, shifts=shift, dims=0)
        outputs = []
        
        for i in range(cycled_tensor.shape[0]):
            # Extract each slice along the first dimension
            tensor_slice = cycled_tensor[i]
            
            # Pass the slice through the model's forward method
            output = self.forward(tensor_slice.unsqueeze(0))  # Unsqueeze to add the batch dimension back
            
            # Store the output
            outputs.append(output)
        
        # Combine the outputs back into a single tensor if needed
        combined_output = torch.cat(outputs, dim=0)
        
        return combined_output


    def forward(self, x):
        # print(x.shape)
        x = self.fc0(x)
        idn = x
        x = self.conv1(x)

        b, c, h, w = x.size()
        n = h*w
        x = x.view(b, c, n)   # b * c * n 

        attn = self.linear_0(x) # b, k, n
        attn = F.softmax(attn, dim=-1) # b, k, n

        attn = attn / (1e-9 + attn.sum(dim=1, keepdim=True)) #  # b, k, n
        x = self.linear_1(attn) # b, c, n

        x = x.view(b, c, h, w)
        x = self.conv2(x)
        x = x + idn
        x = F.relu(x)
        x = self.fc1(x)
        x = self.fc2(x)
        x = x.view(1, -1)
        # print(x.shape)
        x = self.final_linear(x)

        return x
    
class ExternalRPNClip(nn.Module):
    def __init__(self, clip_model, clip_preprocess, anchors_scales, anchor_ratios, feature_map_channels=2048, device = 'cpu'):
        super(ExternalRPNClip, self).__init__()
        self.yolo = YOLO("yolov8n.pt")
        self.EA = External_attention(512).to(device)
        self.clip_model, self.clip_preprocess = clip_model, clip_preprocess
        self.clip_model.visual.attnpool = torch.nn.Identity() # Remove last attention layer of Clip - Resnet50
        self.classifier = self.clip_model.visual.layer4
        # Prepare the Region proposal network with it's anchors and the RPN head
        # self.anchor_generator = AnchorGenerator(anchors_scales, anchor_ratios)
        # self.rpn_head = RPNHead(feature_map_channels, len(self.anchor_generator))

        # rpn_pre_post_nms_top_n = {"training": 200, "testing": 100}
        # self.rpn_wrapper = rpn.RegionProposalNetwork(
        #     self.anchor_generator, self.rpn_head,
        #     fg_iou_thresh=0.7,
        #     bg_iou_thresh=0.2,          # anchor boxes with a IOU < 0.2 are considered negative
        #     batch_size_per_image=256,   # for each image 256 anchors are sampled
        #     positive_fraction=0.5,      # out of the 256 anchors half are positive and half negative
        #     pre_nms_top_n=rpn_pre_post_nms_top_n,
        #     post_nms_top_n=rpn_pre_post_nms_top_n,
        #     nms_thresh=0.7              # threshold for Non maximum suppression
        # ).to(device)
        
        self.rpn = RegionProposalNetwork(
            2048, 1024,
            ratios=ratios,
            anchor_scales=[8, 16, 32],
            feat_stride=32,  # Scaling factor from input size to features map
        )

        self.head = VGG16RoIHead(
            n_class=80 + 1, # +1 is background
            roi_size=7,
            spatial_scale=(1. / 32),
            classifier=self.classifier
        )

        self.faster_rcnn = FasterRCNN(extractor=self.encode_image,
                                 rpn=self.rpn,
                                 head=self.head
                                )

        self.proposal_target_creator = ProposalTargetCreator()
        self.anchor_target_creator = AnchorTargetCreator()

        self.loc_normalize_mean = self.faster_rcnn.loc_normalize_mean
        self.loc_normalize_std = self.faster_rcnn.loc_normalize_std

        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        self.processing = []
        self.similarities = []
        self.device = device

    def infer_bboxes(self, image_path):
        results = self.yolo(image_path, verbose=False)
        # print(results[0])
        bboxes = results[0].boxes.xyxy
        cls = results[0].boxes.cls
        return bboxes,cls

    def encode_image(self, image):
        # Encode the image using the CLIP model
        with torch.no_grad():
            image_features = self.clip_model.encode_image(image)

        return image_features
    
    def preprocess_images(self, cropped_images):
        # preprocess with CLIP each cropped PIL image(converts each image in a image
        # of size [3,224,224])
        processing = []
        for image in cropped_images:
            processed_img = self.clip_preprocess(image).to(self.device)
            # proc = self.clip_model.encode_image(processed_img)
            processing.append(processed_img)
        preprocessed = torch.tensor(np.stack(processing))
        # preprocessed = torch.stack(processing).to(self.device) # return a single tensor
        return preprocessed


    def encode_text(self, text):
        with torch.no_grad():
            text_features = self.clip_model.encode_text(text)
        return text_features
    
    # def hook_fn(self, module, input, output):
    #     self.al = output
    #     self.al.to(device)

    def _smooth_l1_loss(self, x, t, in_weight, sigma):
        sigma2 = sigma ** 2
        diff = in_weight * (x - t)
        abs_diff = diff.abs()
        flag = (abs_diff.data < (1. / sigma2)).float()
        y = (flag * (sigma2 / 2.) * (diff ** 2) +
            (1 - flag) * (abs_diff - 0.5 / sigma2))
        return y.sum()

    def _fast_rcnn_loc_loss(self, pred_loc, gt_loc, gt_label, sigma):
        #in_weight = t.zeros(gt_loc.shape).cuda()
        in_weight = t.zeros(gt_loc.shape)
        # Localization loss is calculated only for positive rois.
        # NOTE:  unlike origin implementation, 
        # we don't need inside_weight and outside_weight, they can calculate by gt_label
        #in_weight[(gt_label > 0).view(-1, 1).expand_as(in_weight).cuda()] = 1
        in_weight[(gt_label > 0).view(-1, 1).expand_as(in_weight)] = 1
        loc_loss = self._smooth_l1_loss(pred_loc, gt_loc, in_weight.detach(), sigma)
        # Normalize by total number of negtive and positive rois.
        loc_loss /= ((gt_label >= 0).sum().float()) # ignore gt_label==-1 for rpn_loss
        return loc_loss

    def forward(self, images, text, gtBoxes, len_im):
        # print(images.shape)
        #print(images)
        #bl = 0 # input of layer4 of CLIP's ResNet
        #hook_handle = clip_model.visual.layer4.register_forward_hook(self.hook_fn) # handle to retrieve output
        #image_features_clip = self.encode_image(images)
        
        image_features_rcnn = self.faster_rcnn.extractor(images)
        #print("FROM CLIP: \n", image_features_clip)
        #print("FROM F-RCNN: \n", image_features_rcnn)

        _, _, H, W = images.shape
        img_size = (H, W)
        #print("CIAO: ", img_size)


        rpn_locs, rpn_scores, rois, roi_indices, anchor = self.faster_rcnn.rpn(image_features_rcnn, img_size, scale=1)

        proposed_bboxes = torch.tensor(rois)

        # calculate the IOU
        best_proposals = torch.zeros((len_im, 4), device="cpu")
        for j in range(len_im):
            # The proposal with the highest score(the first) is the final one
            best_proposals[j,:] = proposed_bboxes[j][0]

        cumulative_accuracy = computeSumIou(gts, best_proposals).item()
        print("cumulative_acc_RPN", cumulative_accuracy)



        sample_roi, gt_roi_loc, gt_roi_label = self.proposal_target_creator(
            rois,
            tonumpy(gtBoxes),
            tonumpy(text),
            self.loc_normalize_mean,
            self.loc_normalize_std)
        
        roi_cls_loc, roi_score = self.faster_rcnn.head(
            image_features_rcnn,
            sample_roi,
            roi_indices)
        

        # ------------------ RPN losses -------------------#
        gt_rpn_loc, gt_rpn_label = self.anchor_target_creator(
            tonumpy(gtBoxes),
            anchor,
            img_size)
        gt_rpn_label = totensor(gt_rpn_label).long()
        gt_rpn_loc = totensor(gt_rpn_loc)
        rpn_loc_loss = self._fast_rcnn_loc_loss(
            rpn_locs,
            gt_rpn_loc,
            gt_rpn_label.data,
            self.rpn_sigma)
        
        #rpn_cls_loss = F.cross_entropy(rpn_scores, gt_rpn_label.cuda(), ignore_index=-1)
        rpn_cls_loss = F.cross_entropy(rpn_scores, gt_rpn_label, ignore_index=-1)
        

        # ------------------ ROI losses (fast rcnn loss) -------------------#
        n_sample = roi_cls_loc.shape[0]
        roi_cls_loc = roi_cls_loc.view(n_sample, -1, 4)
        # roi_loc = roi_cls_loc[t.arange(0, n_sample).long().cuda(), \
        #                       totensor(gt_roi_label).long()]
        roi_loc = roi_cls_loc[t.arange(0, n_sample).long(), \
                              totensor(gt_roi_label).long()]
        gt_roi_label = totensor(gt_roi_label).long()
        gt_roi_loc = totensor(gt_roi_loc)

        roi_loc_loss = self._fast_rcnn_loc_loss(
            roi_loc.contiguous(),
            gt_roi_loc,
            gt_roi_label.data,
            self.roi_sigma)

        #roi_cls_loss = nn.CrossEntropyLoss()(roi_score, gt_roi_label.cuda())
        roi_cls_loss = nn.CrossEntropyLoss()(roi_score, gt_roi_label)

        losses = [rpn_loc_loss, rpn_cls_loss, roi_loc_loss, roi_cls_loss]
        losses = losses + [sum(losses)]

        print("LOSSES: \n", losses)
        
        # print("rpn_locs \n \n", rpn_locs)
        # print("rpn_scores \n \n", rpn_scores)
        # print("rois \n \n", rois)
        # print("roi_indices \n \n", roi_indices)
        # print("anchor \n \n", anchor)

        # print(70*'-')
        # # print(self.al.shape)
        with torch.autocast(device_type="cpu"):
            image_features = self.EA.process_features(image_features_rcnn)
        # print(image_features)
        # print(70*'-')
        # Region Proposal Network
        ground_truth_boxes = [{"boxes": gtBoxes[batch_i].unsqueeze(0)} for batch_i in range(len_im)]
        image_sizes = ImageList([(640,640)] * len_im)   #batch_size = 16
        
        #print(ground_truth_boxes)

        # with torch.autocast(device_type="cpu"):
        #     proposals_RPN, losses_RPN = self.rpn_wrapper(image_sizes, {"0": self.al}, ground_truth_boxes)
        #print(losses_RPN)
        
         #print("RPN: ", rpn_result)
        # print(70*'-')
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        print("CAVOLO: ", text.shape)
        text_features = self.encode_text(text)
        # temp = []
        # temp.append(text_features)
        # temp.append(text_features)
        # temp.append(text_features)
        # text_features2 = torch.cat(temp, dim=0)
        # print(text_features)
        # print(70*'-')
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        # text_features2 = text_features2 / text_features2.norm(dim=-1, keepdim=True)
        logit_scale = self.logit_scale.exp()

        # Conversion to float32
        image_features = image_features.float()
        text_features = text_features.float()
        
        # normalized featuresim
        similarity = (image_features @ text_features.t())
        # print(similarity.cpu())
        # similarity = (image_features @ text_features2.t())
        # print(similarity.cpu())
        scaled_similarity = logit_scale * similarity
        logits_per_image = scaled_similarity
        logits_per_text = scaled_similarity.t()

        # print(text_features.shape)
        # # cosine similarity as logits
        # logits_per_image = logit_scale * (image_features @ text_features.t())
        # print("Logits = " + str(logits_per_image))
        # logits_per_text = logits_per_image.t()

        # shape = [global_batch_size, global_batch_size]
        # hook_handle.remove()

        return logits_per_image, logits_per_text
    

In [151]:
NUM_EPOCHS = 10
count = 0
iou_threshold = 0.5
loss_meter = MetricMeter(name = 'TrainingEA', threshold=iou_threshold)

running_loc_acc = 0.0
correct_bboxes = 0
overall = 0
semantic = 0
semantic_similarity = 0
running_ga = 0
sem = 0
loc_acc = 0
lr = 0.0001
wd = 0.002
alpha = 1 # to decrease lr over time
model = ExternalRPNClip(clip_model,clip_preprocess, anchors_scales=scales, anchor_ratios=ratios,device='cpu')

In [152]:
def train_Pipeline(model):
    model.clip_model.train()
    model.EA.train()
    #model.rpn_head.train()
    #model.rpn_wrapper.train()

def eval_Pipeline(model):
    model.clip_model.eval()
    model.EA.eval()
    #model.rpn_head.eval()
    #model.rpn_wrapper.eval()

In [153]:
cost = nn.CrossEntropyLoss()
optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=wd)

In [154]:
proposal_target_creator = ProposalTargetCreator()
anchor_target_creator = AnchorTargetCreator()

In [155]:
cumulative_accuracy = 0.0
cumulative_loss = 0.0
overall = 0
errors = 0

test_cumulative_accuracy = 0.0
test_cumulative_loss = 0.0
test_overall = 0
test_errors = 0

for i in range(NUM_EPOCHS):
    loop = tqdm(dataloader_train, position=0, leave=True)
    test_loop = tqdm(dataloader_test, position=0, leave=True)
    train_Pipeline(model)
    for _,data in enumerate(loop):
        #try:
            
            texts = data[0]
            texts = texts.squeeze(1).to(device)
            #images = data[1].to(device)
            gts = data[1]
            clss = data[2]
            filename = data[3]
            images = []
            
            for x in filename:
                temp = Image.open(x)
                image = clip_preprocess(temp).to(device)
                images.append(image)

            images = torch.stack(images)
            optimizer.zero_grad()
            len_im = len(images)# last pass could contain less than 16 images

            # Build Data for training pass
            # since confidence is directly how much bbox and text "resembles" each other 
            #li, lt, proposals_RPN, losses_RPN = model.forward(images, texts, gts, len_im)
            li, lt = model.forward(images, texts, gts, len_im)

            
            # Region Proposal Network
            # Calculate the loss
            #loss_RPN = getLoss(losses_RPN["loss_objectness"], losses_RPN["loss_rpn_box_reg"])
            #avg_loss += loss.item()

            # # calculate the IOU
            # best_proposals = torch.zeros((len_im, 4), device="cpu")
            # for j in range(len_im):
            #     # The proposal with the highest score(the first) is the final one
            #     best_proposals[j,:] = proposals_RPN[j][0]

            # cumulative_accuracy += computeSumIou(gts, best_proposals).item()
            # print("cumulative_acc_RPN", cumulative_accuracy)


            # Construct the ground truth
            ground_truth = torch.arange(len(images),dtype=torch.long,device=device)
            img_loss = cost(li, ground_truth)
            desc_loss = cost(lt, ground_truth)
            loss = (img_loss + desc_loss)/2
            #loss += loss_RPN
            loss.backward()
            optimizer.step()

            # Keep track of loss and accuracy metrics to see epochs progress
            overall += len_im #batch_size
            cumulative_loss += loss.item()

            _, predicted = li.max(dim=1)
            cumulative_accuracy += predicted.eq(ground_truth).sum().item()
            loss = cumulative_loss / overall 
            acc = cumulative_accuracy / overall
            #clip.model.convert_weights(model)
        #except:
        #    print('errore nel training')

    loop.set_description(f"Training \Epoch{i}\ Values => Loss Iter = {loss}, Accuracy = {acc}")
    
    if( i % 3 == 0):
        eval_Pipeline(model)
        with torch.no_grad():
            for _,data in enumerate(test_loop):
                try:
                    texts = data[0]
                    texts = texts.squeeze(1).to(device)
                    #images = data[1].to(device)
                    gts = data[1]
                    clss = data[2]
                    filename = data[3]
                    images = []
                    for x in filename:
                        temp = Image.open(x)
                        image = clip_preprocess(temp).to(device)
                        images.append(image)

                    images = torch.stack(images)
                    #optimizer.zero_grad()
                    len_im = len(images)# last pass could contain less than 16 images

                    # Build Data for training pass
                    # since confidence is directly how much bbox and text "resembles" each other 
                    li, lt, proposals_RPN, losses_RPN = model.forward(images, texts, gts, len_im)

                    print(losses_RPN)
                    # Region Proposal Network
                    # Calculate the loss
                    loss_RPN = getLoss(losses_RPN["loss_objectness"], losses_RPN["loss_rpn_box_reg"])
                    #avg_loss += loss.item()

                    # calculate the IOU
                    best_proposals = torch.zeros((len_im, 4), device="cpu")
                    for j in range(len_im):
                        # The proposal with the highest score(the first) is the final one
                        best_proposals[j,:] = proposals_RPN[j][0]

                    cumulative_accuracy += computeSumIou(gts, best_proposals).item()



                    # Construct the ground truth
                    ground_truth = torch.arange(len(images),dtype=torch.long,device=device)
                    img_loss = cost(li, ground_truth)
                    desc_loss = cost(lt, ground_truth)
                    loss = (img_loss + desc_loss)/2
                    loss += loss_RPN
                    # loss.backward()
                    # optimizer.step()

                    # Keep track of loss and accuracy metrics to see epochs progress
                    overall += len_im #batch_size
                    cumulative_loss += loss.item()

                    _, predicted = li.max(dim=1)
                    cumulative_accuracy += predicted.eq(ground_truth).sum().item()
                    loss = cumulative_loss / overall 
                    acc = cumulative_accuracy / overall
                    #clip.model.convert_weights(model)
                except:
                    print('errore nel testing')

                    test_loop.set_description(f"Testing \Epoch{i}\ Values => Loss Iter = {loss}, Accuracy = {acc}")

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

cumulative_acc_RPN 0.0
SHAPE torch.Size([1533])
torch.Size([128, 4])


RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 1533 but got size 128 for tensor number 1 in the list.

In [ ]:
torch.save(model.state_dict(), 'saves/EARPN.pt')